# GS 2026 - Applied Computer Vision

# Sentinela Fire ACV: classificação e previsão de risco de queimadas com imagens ambientais

**Projeto integrado:** plataforma de monitoramento, classificação e previsão de risco de queimadas usando dados climáticos, dados orbitais e imagens ambientais/satelitais.

**Disciplina:** Applied Computer Vision

**Objetivo de visão computacional:** treinar redes neurais convolucionais criadas do zero para classificar imagens em três classes operacionais:

1. Vegetação úmida - baixo risco
2. Vegetação seca - risco moderado
3. Foco de queimada ou cicatriz recente - alto risco

O dataset deste notebook é montado a partir das três fotos adicionadas na pasta do projeto: `umida.jpg`, `seca.jpg` e `queimada.jpg`. Para permitir treino, validação e teste com CNNs, o notebook extrai recortes e aplica pequenas variações de brilho, contraste, espelhamento, rotação e ruído. A saída do melhor modelo é combinada com variáveis climáticas e orbitais simuladas para gerar um nível final de alerta para defesa civil, órgãos ambientais e equipes de campo.

In [1]:
import sys
!{sys.executable} -m pip install -q tensorflow matplotlib numpy pandas scikit-learn seaborn pillow

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 26.0 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
# ============================================================
# GS - Applied Computer Vision
# Sentinela Fire ACV - CNNs do zero para risco de queimadas
# ============================================================
# Este notebook realiza as etapas pedidas na Global Solution:
# 1. Define o problema de visão computacional e conexão espacial
# 2. Usa as fotos umida.jpg, seca.jpg e queimada.jpg como dataset-base
# 3. Gera recortes aumentados das fotos e metadados climáticos/orbitais
# 4. Realiza pré-processamento e divisão treino/validação/teste
# 5. Treina no mínimo 2 arquiteturas de CNN criadas do zero
# 6. Compara acurácia, loss, matriz de confusão e erros
# 7. Demonstra a solução funcionando com novos recortes das fotos
# ============================================================

import os
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from PIL import Image, ImageEnhance, ImageOps
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

import tensorflow as tf
from tensorflow.keras import layers, models, regularizers
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

warnings.filterwarnings('ignore')

SEED = 42
np.random.seed(SEED)
random.seed(SEED)
tf.random.set_seed(SEED)

IMG_SIZE = 64
N_CLASSES = 3
IMAGENS_POR_CLASSE = 500
TEST_SIZE = 0.15
VAL_SIZE = 0.15
BATCH_SIZE = 64
EPOCHS_PADRAO = 12

NOMES_CLASSES = {
    0: 'baixo_risco_vegetacao_umida',
    1: 'risco_moderado_vegetacao_seca',
    2: 'alto_risco_queimada'
}

NOMES_CURTOS = {
    0: 'Baixo risco',
    1: 'Risco moderado',
    2: 'Alto risco'
}

NOTEBOOK_DIR = Path.cwd()
if not (NOTEBOOK_DIR / 'umida.jpg').exists():
    NOTEBOOK_DIR = Path(r'C:UserseduarDesktopgssentinela-app-acv')

ARQUIVOS_IMAGENS_BASE = {
    0: NOTEBOOK_DIR / 'umida.jpg',
    1: NOTEBOOK_DIR / 'seca.jpg',
    2: NOTEBOOK_DIR / 'queimada.jpg'
}

RESULTADOS = []

print('TensorFlow:', tf.__version__)
print('GPU disponível:', bool(tf.config.list_physical_devices('GPU')))
print('Pasta do notebook:', NOTEBOOK_DIR)
print('Configuração pronta.')

## 1. Definição do problema e conexão com a Indústria Espacial

A plataforma **Sentinela Fire** usa visão computacional para analisar imagens ambientais e recortes compatíveis com monitoramento orbital, apoiando decisões preventivas contra queimadas. Em uma operação real, imagens semelhantes poderiam ser integradas a fontes como Sentinel-2, Landsat, CBERS/INPE ou bases públicas de focos de calor.

Neste notebook, o dataset é construído a partir das três imagens fornecidas pela equipe:

- `umida.jpg`: área com vegetação úmida e presença de água;
- `seca.jpg`: área seca com solo rachado e baixa umidade;
- `queimada.jpg`: área com fogo, fumaça e vegetação queimada.

Como há apenas uma foto-base por classe, o notebook extrai diversos recortes e aplica aumento de dados para criar um conjunto treinável. Essa abordagem torna a entrega reprodutível em um único notebook e demonstra a lógica de visão computacional; em produção, o ideal seria ampliar o dataset com muitas cenas reais por classe para reduzir viés de origem.

In [ ]:
def limitar_uint8(img):
    return np.clip(img, 0, 255).astype(np.uint8)


def carregar_imagens_base():
    imagens_base = {}
    print('=' * 70)
    print('IMAGENS-BASE DO DATASET')
    print('=' * 70)
    for classe, caminho in ARQUIVOS_IMAGENS_BASE.items():
        if not caminho.exists():
            raise FileNotFoundError(f'Arquivo não encontrado: {caminho}')
        img = Image.open(caminho).convert('RGB')
        imagens_base[classe] = img
        print(f'{NOMES_CURTOS[classe]} -> {caminho.name} | tamanho original: {img.size}')
    return imagens_base


def escolher_janela_recorte(img_base, classe):
    largura, altura = img_base.size
    menor_lado = min(largura, altura)
    crop_size = int(np.random.uniform(0.34, 0.72) * menor_lado)
    crop_size = max(180, min(crop_size, menor_lado))

    x_max = max(0, largura - crop_size)
    y_max = max(0, altura - crop_size)

    if classe == 0:
        y_min = 0
        y_limite = y_max
    elif classe == 1:
        # Na foto de seca, a evidência principal está na metade inferior: solo rachado.
        y_min = min(y_max, int(altura * 0.35))
        y_limite = y_max
    else:
        # Na foto de queimada, fogo/fumaça aparecem no centro e na parte inferior.
        y_min = min(y_max, int(altura * 0.12))
        y_limite = min(y_max, int(altura * 0.70))
        if y_limite < y_min:
            y_limite = y_max

    x = np.random.randint(0, x_max + 1) if x_max > 0 else 0
    y = np.random.randint(y_min, y_limite + 1) if y_limite > y_min else y_min
    return x, y, crop_size


def aumentar_recorte(patch):
    if random.random() < 0.5:
        patch = ImageOps.mirror(patch)

    if random.random() < 0.25:
        patch = ImageOps.flip(patch)

    angulo = np.random.uniform(-10, 10)
    cor_media = tuple(np.asarray(patch).reshape(-1, 3).mean(axis=0).astype(int))
    patch = patch.rotate(angulo, resample=Image.Resampling.BILINEAR, fillcolor=cor_media)

    patch = ImageEnhance.Brightness(patch).enhance(np.random.uniform(0.82, 1.18))
    patch = ImageEnhance.Contrast(patch).enhance(np.random.uniform(0.82, 1.25))
    patch = ImageEnhance.Color(patch).enhance(np.random.uniform(0.85, 1.18))
    patch = ImageEnhance.Sharpness(patch).enhance(np.random.uniform(0.8, 1.35))

    arr = np.asarray(patch).astype(np.float32)
    arr += np.random.normal(0, np.random.uniform(1.5, 7.0), arr.shape)
    return limitar_uint8(arr)


def gerar_amostra_da_foto(img_base, classe):
    x, y, crop_size = escolher_janela_recorte(img_base, classe)
    patch = img_base.crop((x, y, x + crop_size, y + crop_size))
    patch = patch.resize((IMG_SIZE, IMG_SIZE), Image.Resampling.BILINEAR)
    return aumentar_recorte(patch)


def gerar_metadados(classe, idx):
    satelites = ['Sentinel-2A', 'Sentinel-2B', 'Landsat-9', 'CBERS-4A']
    if classe == 0:
        temp = np.random.normal(24, 3)
        umidade = np.random.normal(76, 8)
        vento = np.random.normal(9, 3)
        chuva = np.random.normal(42, 12)
        ndvi = np.random.normal(0.72, 0.08)
        nbr = np.random.normal(0.62, 0.08)
        frp = np.random.normal(0.4, 0.4)
    elif classe == 1:
        temp = np.random.normal(33, 4)
        umidade = np.random.normal(35, 9)
        vento = np.random.normal(18, 5)
        chuva = np.random.normal(8, 6)
        ndvi = np.random.normal(0.26, 0.08)
        nbr = np.random.normal(0.18, 0.09)
        frp = np.random.normal(2.5, 1.0)
    else:
        temp = np.random.normal(39, 5)
        umidade = np.random.normal(22, 8)
        vento = np.random.normal(25, 7)
        chuva = np.random.normal(3, 4)
        ndvi = np.random.normal(0.10, 0.08)
        nbr = np.random.normal(-0.14, 0.12)
        frp = np.random.normal(20, 6)

    return {
        'id_amostra': f'patch_{classe}_{idx:04d}',
        'classe': classe,
        'classe_nome': NOMES_CLASSES[classe],
        'foto_base': ARQUIVOS_IMAGENS_BASE[classe].name,
        'temperatura_c': float(np.clip(temp, 10, 48)),
        'umidade_pct': float(np.clip(umidade, 5, 100)),
        'vento_kmh': float(np.clip(vento, 0, 55)),
        'chuva_7d_mm': float(np.clip(chuva, 0, 90)),
        'ndvi': float(np.clip(ndvi, -0.2, 0.95)),
        'nbr': float(np.clip(nbr, -0.6, 0.9)),
        'frp_simulada_mw': float(np.clip(frp, 0, 40)),
        'latitude': float(np.random.uniform(-24.5, -6.0)),
        'longitude': float(np.random.uniform(-62.0, -42.0)),
        'satelite': random.choice(satelites),
        'orbita_km': float(np.random.normal(705, 12)),
        'resolucao_m': random.choice([10, 20, 30]),
        'dia_juliano': int(np.random.randint(1, 366))
    }

In [ ]:
# ============================================================
# 2. Gerar dataset a partir das fotos adicionadas no projeto
# ============================================================

def criar_dataset():
    imagens_base = carregar_imagens_base()
    imagens = []
    labels = []
    metadados = []

    for classe, img_base in imagens_base.items():
        for idx in range(IMAGENS_POR_CLASSE):
            imagens.append(gerar_amostra_da_foto(img_base, classe))
            labels.append(classe)
            metadados.append(gerar_metadados(classe, idx))

    imagens = np.asarray(imagens, dtype=np.uint8)
    labels = np.asarray(labels, dtype=np.int32)
    metadados = pd.DataFrame(metadados)
    return imagens, labels, metadados, imagens_base


X, y, df_meta, imagens_base = criar_dataset()

print('=' * 70)
print('DATASET SENTINELA FIRE ACV')
print('=' * 70)
print('Imagens:', X.shape)
print('Labels:', y.shape)
print('Metadados:', df_meta.shape)
print()
print('Distribuição por classe:')
print(df_meta['classe_nome'].value_counts())
print()
print('Fotos-base usadas por classe:')
display(df_meta[['classe_nome', 'foto_base']].drop_duplicates().sort_values('classe_nome'))
print('Primeiras linhas dos metadados climáticos/orbitais:')
display(df_meta.head())

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for classe, ax in enumerate(axes):
    ax.imshow(imagens_base[classe])
    ax.set_title(f'{NOMES_CURTOS[classe]}: {ARQUIVOS_IMAGENS_BASE[classe].name}')
    ax.axis('off')
plt.suptitle('Fotos originais adicionadas pela equipe')
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(3, 6, figsize=(14, 7))
for classe in range(N_CLASSES):
    idxs = np.where(y == classe)[0]
    selecionados = np.random.choice(idxs, 6, replace=False)
    for col, idx in enumerate(selecionados):
        axes[classe, col].imshow(X[idx])
        axes[classe, col].axis('off')
        if col == 0:
            axes[classe, col].set_title(NOMES_CURTOS[classe], fontsize=11)
plt.suptitle('Recortes aumentados gerados a partir das fotos-base', fontsize=14)
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(1, 4, figsize=(15, 3.5))
variaveis = ['temperatura_c', 'umidade_pct', 'ndvi', 'frp_simulada_mw']
for ax, var in zip(axes, variaveis):
    sns.boxplot(data=df_meta, x='classe_nome', y=var, ax=ax)
    ax.set_xlabel('')
    ax.tick_params(axis='x', rotation=40)
    ax.set_title(var)
plt.suptitle('Coerência dos metadados com as classes visuais')
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# 3. Pré-processamento e divisão treino/validação/teste
# ============================================================

indices = np.arange(len(X))
idx_train, idx_temp, y_train_raw, y_temp_raw = train_test_split(
    indices, y,
    test_size=(TEST_SIZE + VAL_SIZE),
    random_state=SEED,
    stratify=y
)

proporcao_teste_no_temp = TEST_SIZE / (TEST_SIZE + VAL_SIZE)
idx_val, idx_test, y_val_raw, y_test_raw = train_test_split(
    idx_temp, y_temp_raw,
    test_size=proporcao_teste_no_temp,
    random_state=SEED,
    stratify=y_temp_raw
)

x_train = X[idx_train].astype('float32') / 255.0
x_val = X[idx_val].astype('float32') / 255.0
x_test = X[idx_test].astype('float32') / 255.0

y_train = to_categorical(y[idx_train], N_CLASSES)
y_val = to_categorical(y[idx_val], N_CLASSES)
y_test = to_categorical(y[idx_test], N_CLASSES)

df_train = df_meta.iloc[idx_train].reset_index(drop=True)
df_val = df_meta.iloc[idx_val].reset_index(drop=True)
df_test = df_meta.iloc[idx_test].reset_index(drop=True)

print('Treino:', x_train.shape, y_train.shape)
print('Validação:', x_val.shape, y_val.shape)
print('Teste:', x_test.shape, y_test.shape)

for nome, labels in [('Treino', y[idx_train]), ('Validação', y[idx_val]), ('Teste', y[idx_test])]:
    contagem = pd.Series(labels).map(NOMES_CURTOS).value_counts().sort_index()
    print()
    print(f'{nome}:')
    print(contagem)

In [ ]:
# ============================================================
# 4. Criar arquiteturas de CNN do zero
# ============================================================

def criar_modelo_cnn_basica(input_shape=(IMG_SIZE, IMG_SIZE, 3), n_classes=N_CLASSES):
    modelo = models.Sequential(name='CNN_Basica_Sentinela')
    modelo.add(layers.Input(shape=input_shape))
    modelo.add(layers.Conv2D(16, (3, 3), activation='relu', padding='same'))
    modelo.add(layers.MaxPooling2D((2, 2)))
    modelo.add(layers.Conv2D(32, (3, 3), activation='relu', padding='same'))
    modelo.add(layers.MaxPooling2D((2, 2)))
    modelo.add(layers.Conv2D(64, (3, 3), activation='relu', padding='same'))
    modelo.add(layers.MaxPooling2D((2, 2)))
    modelo.add(layers.Flatten())
    modelo.add(layers.Dense(96, activation='relu'))
    modelo.add(layers.Dropout(0.25))
    modelo.add(layers.Dense(n_classes, activation='softmax'))

    modelo.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    return modelo


def criar_modelo_cnn_regularizada(input_shape=(IMG_SIZE, IMG_SIZE, 3), n_classes=N_CLASSES):
    modelo = models.Sequential(name='CNN_Profunda_Regularizada_Sentinela')
    modelo.add(layers.Input(shape=input_shape))

    modelo.add(layers.Conv2D(32, (3, 3), padding='same', activation='relu', kernel_regularizer=regularizers.l2(1e-4)))
    modelo.add(layers.Conv2D(32, (3, 3), padding='same', activation='relu', kernel_regularizer=regularizers.l2(1e-4)))
    modelo.add(layers.MaxPooling2D((2, 2)))
    modelo.add(layers.Dropout(0.15))

    modelo.add(layers.Conv2D(64, (3, 3), padding='same', activation='relu', kernel_regularizer=regularizers.l2(1e-4)))
    modelo.add(layers.Conv2D(64, (3, 3), padding='same', activation='relu', kernel_regularizer=regularizers.l2(1e-4)))
    modelo.add(layers.MaxPooling2D((2, 2)))
    modelo.add(layers.Dropout(0.25))

    modelo.add(layers.Conv2D(128, (3, 3), padding='same', activation='relu', kernel_regularizer=regularizers.l2(1e-4)))
    modelo.add(layers.MaxPooling2D((2, 2)))
    modelo.add(layers.Flatten())
    modelo.add(layers.Dense(128, activation='relu'))
    modelo.add(layers.Dropout(0.35))
    modelo.add(layers.Dense(n_classes, activation='softmax'))

    modelo.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.0008),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    return modelo


modelo_basico = criar_modelo_cnn_basica()
modelo_regularizado = criar_modelo_cnn_regularizada()

print('Arquitetura 1 - CNN Básica')
modelo_basico.summary()
print()
print('Arquitetura 2 - CNN Regularizada')
modelo_regularizado.summary()

In [ ]:
# ============================================================
# 5. Treinar e avaliar os modelos
# ============================================================

def treinar_e_avaliar(modelo, descricao, epochs=EPOCHS_PADRAO):
    print()
    print('=' * 70)
    print(f'Treinando: {descricao}')
    print('=' * 70)

    callbacks = [
        EarlyStopping(monitor='val_loss', patience=4, restore_best_weights=True),
        ReduceLROnPlateau(monitor='val_loss', patience=2, factor=0.5, min_lr=1e-5)
    ]

    historico = modelo.fit(
        x_train, y_train,
        validation_data=(x_val, y_val),
        epochs=epochs,
        batch_size=BATCH_SIZE,
        callbacks=callbacks,
        verbose=1
    )

    loss, acc = modelo.evaluate(x_test, y_test, verbose=0)
    print(f'Acurácia no teste: {acc:.4f} ({acc*100:.2f}%)')
    print(f'Loss no teste: {loss:.4f}')

    RESULTADOS.append({
        'descricao': descricao,
        'modelo': modelo,
        'historico': historico,
        'epochs_executadas': len(historico.history['loss']),
        'acuracia': float(acc),
        'loss': float(loss),
        'parametros': int(modelo.count_params())
    })
    return modelo, historico, acc


modelo_basico, hist_basico, acc_basico = treinar_e_avaliar(
    modelo_basico,
    'CNN Básica - 3 blocos convolucionais + Flatten',
    epochs=EPOCHS_PADRAO
)

modelo_regularizado, hist_regularizado, acc_regularizado = treinar_e_avaliar(
    modelo_regularizado,
    'CNN Profunda Regularizada - filtros maiores + Dropout',
    epochs=EPOCHS_PADRAO
)

In [ ]:
# ============================================================
# 6. Comparação entre arquiteturas
# ============================================================

df_resultados = pd.DataFrame([
    {
        'modelo': r['descricao'],
        'epochs_executadas': r['epochs_executadas'],
        'acuracia_teste': r['acuracia'],
        'loss_teste': r['loss'],
        'parametros': r['parametros']
    }
    for r in RESULTADOS
]).sort_values('acuracia_teste', ascending=False)

display(df_resultados)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for r in RESULTADOS:
    hist = r['historico'].history
    axes[0].plot(hist['accuracy'], label=f"{r['descricao']} - treino")
    axes[0].plot(hist['val_accuracy'], linestyle='--', label=f"{r['descricao']} - val")
    axes[1].plot(hist['loss'], label=f"{r['descricao']} - treino")
    axes[1].plot(hist['val_loss'], linestyle='--', label=f"{r['descricao']} - val")

axes[0].set_title('Evolução da acurácia')
axes[0].set_xlabel('Época')
axes[0].set_ylabel('Acurácia')
axes[0].legend(fontsize=8)
axes[0].grid(alpha=0.25)

axes[1].set_title('Evolução da loss')
axes[1].set_xlabel('Época')
axes[1].set_ylabel('Loss')
axes[1].legend(fontsize=8)
axes[1].grid(alpha=0.25)
plt.tight_layout()
plt.show()

plt.figure(figsize=(8, 4))
cores = ['#2E7D32' if acc >= 0.88 else '#C62828' for acc in df_resultados['acuracia_teste']]
bars = plt.barh(df_resultados['modelo'], df_resultados['acuracia_teste'], color=cores)
plt.axvline(0.88, color='black', linestyle='--', label='Referência mínima 88%')
plt.xlim(0, 1.02)
plt.xlabel('Acurácia no teste')
plt.title('Comparação de desempenho das CNNs')
for bar in bars:
    valor = bar.get_width()
    plt.text(valor + 0.01, bar.get_y() + bar.get_height()/2, f'{valor:.2%}', va='center')
plt.legend()
plt.tight_layout()
plt.show()

melhor_resultado = max(RESULTADOS, key=lambda item: item['acuracia'])
melhor_modelo = melhor_resultado['modelo']
print('Melhor modelo:', melhor_resultado['descricao'])
print(f"Acurácia: {melhor_resultado['acuracia']:.4f}")

if melhor_resultado['acuracia'] >= 0.88:
    print('Meta de 88% atingida no conjunto de teste.')
else:
    print('Meta de 88% não atingida. Possíveis melhorias: aumentar o dataset, enriquecer texturas, ajustar regularização e treinar por mais épocas.')

In [ ]:
# ============================================================
# 7. Matriz de confusão e relatório de classificação
# ============================================================

y_prob = melhor_modelo.predict(x_test, verbose=0)
y_pred = np.argmax(y_prob, axis=1)
y_real = y[idx_test]

cm = confusion_matrix(y_real, y_pred)
acc_final = accuracy_score(y_real, y_pred)

print(f'Acurácia final do melhor modelo: {acc_final:.4f} ({acc_final*100:.2f}%)')
print()
print('Relatório de classificação:')
print(classification_report(
    y_real,
    y_pred,
    target_names=[NOMES_CURTOS[i] for i in range(N_CLASSES)]
))

plt.figure(figsize=(7, 5.5))
sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='YlGnBu',
    xticklabels=[NOMES_CURTOS[i] for i in range(N_CLASSES)],
    yticklabels=[NOMES_CURTOS[i] for i in range(N_CLASSES)]
)
plt.title('Matriz de confusão - melhor CNN')
plt.xlabel('Classe predita')
plt.ylabel('Classe real')
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# 8. Análise qualitativa: acertos e erros
# ============================================================

def plotar_exemplos(indices, titulo, max_itens=8):
    if len(indices) == 0:
        print(f'{titulo}: nenhum exemplo encontrado.')
        return

    selecionados = np.random.choice(indices, min(max_itens, len(indices)), replace=False)
    cols = 4
    rows = int(np.ceil(len(selecionados) / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(13, 3.4 * rows))
    axes = np.array(axes).reshape(-1)

    for ax, idx_local in zip(axes, selecionados):
        ax.imshow(x_test[idx_local])
        real = NOMES_CURTOS[y_real[idx_local]]
        pred = NOMES_CURTOS[y_pred[idx_local]]
        conf = y_prob[idx_local, y_pred[idx_local]]
        ax.set_title(f'Real: {real} Pred: {pred} ({conf:.1%})', fontsize=10)
        ax.axis('off')

    for ax in axes[len(selecionados):]:
        ax.axis('off')

    plt.suptitle(titulo, fontsize=14)
    plt.tight_layout()
    plt.show()


idx_acertos = np.where(y_real == y_pred)[0]
idx_erros = np.where(y_real != y_pred)[0]

print(f'Total de acertos: {len(idx_acertos)} / {len(y_real)}')
print(f'Total de erros: {len(idx_erros)} / {len(y_real)}')

plotar_exemplos(idx_acertos, 'Exemplos de acertos do modelo')
plotar_exemplos(idx_erros, 'Exemplos de erros do modelo')

if len(idx_erros) > 0:
    df_erros = pd.DataFrame({
        'classe_real': [NOMES_CURTOS[i] for i in y_real[idx_erros]],
        'classe_predita': [NOMES_CURTOS[i] for i in y_pred[idx_erros]],
        'confianca_predicao': np.max(y_prob[idx_erros], axis=1)
    })
    print()
    print('Resumo dos erros:')
    display(df_erros.groupby(['classe_real', 'classe_predita']).agg(
        quantidade=('confianca_predicao', 'count'),
        confianca_media=('confianca_predicao', 'mean')
    ).reset_index())

In [ ]:
# ============================================================
# 9. Demonstração funcional: nova imagem + previsão de risco
# ============================================================

def gerar_nova_cena(tipo='auto'):
    mapa = {
        'baixo': 0,
        'moderado': 1,
        'alto': 2
    }
    if tipo == 'auto':
        tipo = random.choice(list(mapa.keys()))
    classe_base = mapa[tipo]
    img = gerar_amostra_da_foto(imagens_base[classe_base], classe_base)
    meta = gerar_metadados(classe_base, np.random.randint(9999))
    return img, meta


def normalizar_intervalo(valor, minimo, maximo):
    return float(np.clip((valor - minimo) / (maximo - minimo), 0, 1))


def calcular_alerta_operacional(probabilidades, meta):
    prob_moderado = float(probabilidades[1])
    prob_alto = float(probabilidades[2])
    temp_norm = normalizar_intervalo(meta['temperatura_c'], 18, 45)
    baixa_umidade = 1 - normalizar_intervalo(meta['umidade_pct'], 15, 90)
    vento_norm = normalizar_intervalo(meta['vento_kmh'], 0, 45)
    seca_norm = 1 - normalizar_intervalo(meta['chuva_7d_mm'], 0, 60)
    ndvi_baixo = 1 - normalizar_intervalo(meta['ndvi'], -0.1, 0.85)
    frp_norm = normalizar_intervalo(meta['frp_simulada_mw'], 0, 30)

    score = (
        0.40 * prob_alto +
        0.24 * prob_moderado +
        0.12 * temp_norm +
        0.08 * baixa_umidade +
        0.07 * vento_norm +
        0.05 * seca_norm +
        0.03 * ndvi_baixo +
        0.01 * frp_norm
    )

    if score >= 0.70:
        nivel = 'Crítico'
        acao = 'acionar alerta imediato e priorizar verificação de campo'
    elif score >= 0.55:
        nivel = 'Alto'
        acao = 'monitorar em alta frequência e preparar equipe preventiva'
    elif score >= 0.35:
        nivel = 'Moderado'
        acao = 'manter observação e revisar previsão climática nas próximas horas'
    else:
        nivel = 'Baixo'
        acao = 'monitoramento de rotina'

    return score, nivel, acao


def prever_cena(img, meta, titulo='Cena nova'):
    entrada = img.astype('float32')[None, ...] / 255.0
    probs = melhor_modelo.predict(entrada, verbose=0)[0]
    classe_pred = int(np.argmax(probs))
    score, nivel, acao = calcular_alerta_operacional(probs, meta)

    print('=' * 70)
    print(titulo)
    print('=' * 70)
    print('Classe visual predita:', NOMES_CURTOS[classe_pred])
    print('Probabilidades por classe:')
    for i, p in enumerate(probs):
        print(f'  {NOMES_CURTOS[i]}: {p:.2%}')
    print()
    print(f'Score operacional de risco: {score:.3f}')
    print('Nível de alerta:', nivel)
    print('Ação recomendada:', acao)
    print()
    print('Metadados climáticos/orbitais da cena:')
    display(pd.DataFrame([meta]))

    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    axes[0].imshow(img)
    axes[0].set_title(f'{titulo} Predição: {NOMES_CURTOS[classe_pred]}')
    axes[0].axis('off')

    axes[1].bar([NOMES_CURTOS[i] for i in range(N_CLASSES)], probs, color=['#2E7D32', '#F9A825', '#C62828'])
    axes[1].set_ylim(0, 1)
    axes[1].set_ylabel('Probabilidade')
    axes[1].set_title(f'Alerta operacional: {nivel}')
    axes[1].tick_params(axis='x', rotation=25)
    plt.tight_layout()
    plt.show()

    return {
        'classe_predita': NOMES_CURTOS[classe_pred],
        'score_risco': score,
        'nivel_alerta': nivel,
        'acao_recomendada': acao
    }


cenarios_demo = [('baixo', 'Cena 1 - área úmida'), ('moderado', 'Cena 2 - vegetação seca'), ('alto', 'Cena 3 - foco de queimada')]
respostas_demo = []
for tipo, titulo in cenarios_demo:
    img_demo, meta_demo = gerar_nova_cena(tipo)
    respostas_demo.append(prever_cena(img_demo, meta_demo, titulo=titulo))

display(pd.DataFrame(respostas_demo))